In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# Import the 5 required Machine Learning models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

DATA_FILE = "wdbc.data"
TEST_CSV_FILE = "test_data.csv"

def generate_mismatched_test_csv(df_source):
    """Generates test_data.csv with spaces and 'error' instead of underscores and '_se'."""
    _, df_test = train_test_split(df_source, test_size=0.2, random_state=42, stratify=df_source['diagnosis'])
    
    new_cols = []
    for col in df_test.columns:
        if col in ['id', 'diagnosis']:
            new_cols.append(col)
        else:
            c = col.replace('_se', ' error').replace('_', ' ')
            new_cols.append(c)
            
    df_test.columns = new_cols
    df_test.to_csv(TEST_CSV_FILE, index=False)
    print(f"✅ Generated '{TEST_CSV_FILE}' with mismatched column names (spaces & 'error') and {len(df_test)} rows.")

def run_pipeline():
    if not os.path.exists(DATA_FILE):
        print(f"❌ Error: '{DATA_FILE}' not found in the current directory.")
        return
        
    # Define exact descriptive feature names for the 32-column WDBC dataset
    column_names = [
        'id', 'diagnosis',
        'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
        'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
        'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
        'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se',
        'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
        'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst'
    ]
    
    # Read the master dataset
    df = pd.read_csv(DATA_FILE, header=None, names=column_names)
    
    # Step 1: Generate the test CSV file dynamically
    generate_mismatched_test_csv(df)
    
    # Step 2: Preprocess master dataset for training
    df_ml = df.drop(columns=['id'])
    le = LabelEncoder()
    df_ml['diagnosis'] = le.fit_transform(df_ml['diagnosis']) # M -> 1, B -> 0
    
    X = df_ml.drop(columns=['diagnosis'])
    y = df_ml['diagnosis']
    
    X_train, X_test_base, y_train, y_test_base = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    
    # Step 3: Load and clean the generated test data CSV
    test_df_raw = pd.read_csv(TEST_CSV_FILE)
    
    y_eval = None
    if 'diagnosis' in test_df_raw.columns:
        diag_col = test_df_raw['diagnosis']
        # Safely convert B/M strings to 0/1 regardless of pandas string vs object dtype
        if diag_col.dtype == object or diag_col.astype(str).str.contains('B|M').any():
            # If values are literally string letters 'B' or 'M'
            y_eval = diag_col.map({'B': 0, 'M': 1, 'b': 0, 'm': 1}).values
            if pd.isna(y_eval).any():  # Fallback to LabelEncoder if map missed anything
                y_eval = le.transform(diag_col.astype(str))
        else:
            y_eval = diag_col.values.astype(int)
        test_df_raw = test_df_raw.drop(columns=['diagnosis'])
        
    if 'id' in test_df_raw.columns:
        test_df_raw = test_df_raw.drop(columns=['id'])
        
    # Clean the column names of the test dataframe to match training format
    test_df_raw.columns = test_df_raw.columns.str.strip().str.lower()
    test_df_raw.columns = test_df_raw.columns.str.replace(' ', '_')
    test_df_raw.columns = test_df_raw.columns.str.replace('_error', '_se')
    
    X_eval_scaled = pd.DataFrame(scaler.transform(test_df_raw), columns=X.columns)
    
    # Fallback to test split integers if target values are missing/mismatched
    if y_eval is None or len(y_eval) != len(test_df_raw):
        y_eval = y_test_base.values
    else:
        y_eval = np.array(y_eval, dtype=int)

    # Step 4: Initialize and train models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=10000, random_state=42),
        "Decision Tree Classifier": DecisionTreeClassifier(random_state=42),
        "K-Nearest Neighbor Classifier": KNeighborsClassifier(n_neighbors=5),
        "Naive Bayes Classifier (Gaussian)": GaussianNB(),
        "Ensemble Model (Random Forest)": RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    metrics_summary = []
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_eval_scaled)
        
        try:
            y_prob = model.predict_proba(X_eval_scaled)[:, 1]
            auc = roc_auc_score(y_eval, y_prob)
        except (AttributeError, ValueError):
            auc = 0.0

        metrics_summary.append({
            "Model Name": name,
            "Accuracy": round(accuracy_score(y_eval, y_pred), 4),
            "AUC Score": round(auc, 4),
            "Precision": round(precision_score(y_eval, y_pred, zero_division=0), 4),
            "Recall": round(recall_score(y_eval, y_pred, zero_division=0), 4),
            "F1 Score": round(f1_score(y_eval, y_pred, zero_division=0), 4),
            "MCC Score": round(matthews_corrcoef(y_eval, y_pred), 4)
        })
        
    # Print results scorecard
    results_df = pd.DataFrame(metrics_summary)
    print("\n" + "="*85)
    print("COMPARATIVE EVALUATION PERFORMANCE MATRIX (PROCESSED FROM MISMATCHED CSV)")
    print("="*85)
    print(results_df.to_string(index=False))
    print("="*85)

if __name__ == "__main__":
    run_pipeline()


✅ Generated 'test_data.csv' with mismatched column names (spaces & 'error') and 114 rows.


/lib/python3.14/site-packages/threadpoolctl.py:1135: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():



COMPARATIVE EVALUATION PERFORMANCE MATRIX (PROCESSED FROM MISMATCHED CSV)
                       Model Name  Accuracy  AUC Score  Precision  Recall  F1 Score  MCC Score
              Logistic Regression    0.9649     0.9960     0.9750  0.9286    0.9512     0.9245
         Decision Tree Classifier    0.9298     0.9246     0.9048  0.9048    0.9048     0.8492
    K-Nearest Neighbor Classifier    0.9561     0.9823     0.9744  0.9048    0.9383     0.9058
Naive Bayes Classifier (Gaussian)    0.9211     0.9891     0.9231  0.8571    0.8889     0.8292
   Ensemble Model (Random Forest)    0.9737     0.9929     1.0000  0.9286    0.9630     0.9442
